###**Libraries & Setup**

In [2]:
import os
os.environ["WANDB_MODE"] = "disabled"

import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    util
)


### **Load Dataset (CSV)**

In [3]:
df = pd.read_csv("/content/Resume.csv")
print(df.head())
print(df.columns)


         ID                                         Resume_str  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   
2  33176873           HR DIRECTOR       Summary      Over 2...   
3  27018550           HR SPECIALIST       Summary    Dedica...   
4  17812897           HR MANAGER         Skill Highlights  ...   

                                         Resume_html Category  
0  <div class="fontsize fontface vmargins hmargin...       HR  
1  <div class="fontsize fontface vmargins hmargin...       HR  
2  <div class="fontsize fontface vmargins hmargin...       HR  
3  <div class="fontsize fontface vmargins hmargin...       HR  
4  <div class="fontsize fontface vmargins hmargin...       HR  
Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')


### **Basic EDA**

In [4]:
print("Total samples:", len(df))
print(df['Category'].value_counts())
print(df.isnull().sum())


Total samples: 2484
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64
ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64


### **Preprocessing**

In [5]:
df['Resume_str'] = df['Resume_str'].astype(str)
df = df.dropna(subset=['Resume_str'])


### **DATA SPLITTING (80 / 10 / 10)**

In [6]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Train: 1987
Validation: 248
Test: 249


### **MODEL ARCHITECTURE**

In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print(" Model loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Model loaded


### Training Data

In [8]:
train_examples = [
    InputExample(texts=[text, text], label=1.0)
    for text in train_df['Resume_str'].tolist()
]


In [9]:
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16
)

train_loss = losses.CosineSimilarityLoss(model)


### **TRAINING (EPOCHS = 10**

In [10]:
EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    print(f"\n Epoch {epoch}/{EPOCHS}")

    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=1,
        warmup_steps=100,
        show_progress_bar=True
    )

    print(f" Epoch {epoch} completed")



 Epoch 1/10


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


Step,Training Loss


 Epoch 1 completed

 Epoch 2/10


...

 Epoch 8 completed

 Epoch 9/10


Step,Training Loss


 Epoch 9 completed

 Epoch 10/10


Step,Training Loss


 Epoch 10 completed


### **Model Save**

In [11]:
model.save("minilm_resume_matcher")
print(" Model saved")


 Model saved


### **Evaluation(Test Set)**

In [12]:
test_embeddings = model.encode(
    test_df['Resume_str'].tolist(),
    convert_to_tensor=True
)


In [13]:
def evaluate(job_text, top_k=5):
    job_emb = model.encode(job_text, convert_to_tensor=True)
    scores = util.cos_sim(job_emb, test_embeddings)[0]
    vals, idxs = scores.topk(top_k)

    for v, i in zip(vals, idxs):
        print(
            f"ID: {test_df.iloc[i]['ID']} | "
            f"Category: {test_df.iloc[i]['Category']} | "
            f"Match: {round(float(v)*100,2)}%"
        )


### **MODEL EVALUATION (Validation Set)**

In [16]:
val_texts = val_df['Resume_str'].tolist()

val_embeddings = model.encode(
    val_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

### **Validation embeddings**

In [17]:
from sentence_transformers import util

similarity_matrix = util.cos_sim(val_embeddings, val_embeddings)

print("Similarity matrix shape:", similarity_matrix.shape)


Similarity matrix shape: torch.Size([248, 248])


### **Similarity matrix (rough evaluation)**

In [18]:
test_texts = test_df['Resume_str'].tolist()

test_embeddings = model.encode(
    test_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

test_similarity = util.cos_sim(test_embeddings, test_embeddings)
print("Test similarity computed")


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Test similarity computed


### **Testing (Unseen Data)**

In [19]:
resume_1 = test_texts[0]
resume_2 = test_texts[1]

emb = model.encode([resume_1, resume_2], convert_to_tensor=True)
score = util.cos_sim(emb[0], emb[1])

print("Similarity score:", float(score))


Similarity score: 0.9998074173927307


### **Model Save**

In [20]:
model.save("/content/resume_match_model")
print("Model saved successfully")


Model saved successfully


### **Deployment_Gradio App**

### **Resume Matching Function**

In [21]:
def resume_matcher(resume_text, job_text):
    embeddings = model.encode([resume_text, job_text])
    similarity = util.cos_sim(embeddings[0], embeddings[1])
    return round(float(similarity), 3)


### **Gradio Interface**

In [22]:
import gradio as gr

interface = gr.Interface(
    fn=resume_matcher,
    inputs=[
        gr.Textbox(lines=10, label="Resume Text"),
        gr.Textbox(lines=10, label="Job Description")
    ],
    outputs=gr.Number(label="Matching Score"),
    title="Resume–Job Matching System",
    description="Transformer-based Resume Matching using Sentence-BERT"
)

interface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5572af2ca5e64ab4d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
